In [1]:
!pip install -U pip
!pip install -U accelerate bitsandbytes huggingface_hub
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U pillow sentencepiece timm ftfy
!pip install -U bert-score rouge-score nltk underthesea einops
!pip install git+https://github.com/huggingface/transformers
!pip install transformers==4.57.0



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.9 MB/s eta 0:00:0000:0100:010m
  Attempting uninstall: pip
    Found existing installation: pip 24.0
ERROR: Cannot uninstall pip 24.0, RECORD file not found. Hint: The package was installed by debian.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 521.0/521.0 kB 1.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.4/201.4 kB 821.0 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 10.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.2/507.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 3.3 MB/s eta 0:00:0000:01m0:03m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 5.5 MB/s eta 0:00:00:00:010:02m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [5]:
import zipfile, shutil
from pathlib import Path

zip_path = Path("images.zip")   # tên file zip
extract_dir = Path("temp_extract")

# 1. Giải nén vào thư mục tạm
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# 2. Nếu đã có thư mục images cũ thì xóa đi
shutil.rmtree("images", ignore_errors=True)

# 3. Đổi tên/thay thế thành "images"
extract_dir.rename("images")

print("Ảnh đã nằm trong thư mục ./images")


Ảnh đã nằm trong thư mục ./images


# Load Model


In [2]:
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
import torch
import os

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4", 
    bnb_4bit_compute_dtype="bfloat16",
)
model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    device_map="auto",
    quantization_config=bnb_config,
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

# No Instruction 


In [6]:
import json
from pathlib import Path
from PIL import Image
import torch
from transformers import AutoProcessor, AutoTokenizer
from bert_score import score as bertscore
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
import shutil
import nltk
import re

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# --- DATA PATHS ---
data_dir = Path('.')
images_dir = data_dir / 'images' / 'images'
chart_jsonl = data_dir / 'Chart_data.jsonl'
#table_jsonl = data_dir / 'Table_data.jsonl'
checkpoint_dir = Path('results_batch.jsonl')

# --- INITIALIZE TOKENIZER / PROCESSOR / MODEL 
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

# --- Normalization utilities
def strip_trailing_punct(s):
    """
    Mimic the Kaggle cell behavior: trim whitespace and remove common trailing punctuation.
    """
    if s is None:
        return ""
    s = str(s).strip()
    # Remove surrounding brackets if present
    s = s.strip('()[]{}')
    # Remove trailing punctuation including some fullwidth punctuation and invisible marks
    s = s.rstrip(" .，。,;:!?\"'\u3000\uFEFF")
    return s

def normalize_whitespace(s):
    """
    Collapse repeated whitespace (including NBSP and zero-width spaces) into a single space,
    and trim leading/trailing whitespace.
    """
    if s is None:
        return ""
    s = str(s)
    s = s.replace('\u00A0', ' ')  # NBSP
    s = s.replace('\u200b', '')   # zero width space
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def clean_text_for_metrics(s):
    """
    Combine whitespace normalization and trailing-punctuation stripping,
    returning a normalized string suitable for metrics computation.
    """
    if s is None:
        return ""
    s = normalize_whitespace(s)
    s = strip_trailing_punct(s)
    s = normalize_whitespace(s)
    return s

# --- READ JSONL FILE ---
def reconstruct_and_load_jsonl(file_path):
    data = []
    current_json = ""
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            current_json += line
            try:
                json_obj = json.loads(current_json)
                data.append(json_obj)
                current_json = ""
            except json.JSONDecodeError:
                continue
    return data

# --- IMAGE PREPROCESSING ---
def find_closest_aspect_ratio(aspect_ratio, target_ratios):
    min_diff = float('inf')
    best_ratio = (1, 1)
    for ratio in target_ratios:
        target_ar = ratio[0] / ratio[1]
        diff = abs(aspect_ratio - target_ar)
        if diff < min_diff:
            min_diff = diff
            best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=6, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = set()
    for n in range(min_num, max_num + 1):
        for i in range(1, n + 1):
            for j in range(1, n + 1):
                if i * j <= max_num and i * j >= min_num:
                    target_ratios.add((i, j))
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_aspect_ratio = find_closest_aspect_ratio(aspect_ratio, target_ratios)
    w_ratio, h_ratio = target_aspect_ratio
    target_width = int(image_size * w_ratio)
    target_height = int(image_size * h_ratio)
    num_patches = w_ratio * h_ratio
    resized_img = image.resize((target_width, target_height), Image.Resampling.BICUBIC)
    processed_images = []
    for i in range(int(h_ratio)):
        for j in range(int(w_ratio)):
            left = j * image_size
            upper = i * image_size
            right = left + image_size
            lower = upper + image_size
            patch = resized_img.crop((left, upper, right, lower))
            processed_images.append(patch)
    if use_thumbnail and num_patches != 1:
        thumbnail_img = image.resize((image_size, image_size), Image.Resampling.BICUBIC)
        processed_images.append(thumbnail_img)
    return processed_images, num_patches

def load_image(image_file, input_size=448, max_num=6):
    try:
        image = Image.open(image_file).convert('RGB')
        width, height = 448, 448
        image = image.resize((width, height))
        images, num_patches = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
        return images, num_patches, image
    except Exception as e:
        print(f"Error loading image {image_file}: {e}")
        return None, None, None

# --- CALCULATE METRICS ---
def compute_metrics(pred, ref):
    # Clean both prediction and reference of trailing punctuation and extra whitespace
    pred = "" if pred is None else strip_trailing_punct(pred)
    ref = "" if ref is None else strip_trailing_punct(ref)
    # BERTScore
    try:
        P, R, F1 = bertscore([pred], [ref], lang="vi", rescale_with_baseline=False)
        bertscore_value = float(F1[0])
    except Exception as e:
        print(f"Error computing BERTScore: {e}")
        bertscore_value = 0.0
    # BLEU-1 to BLEU-4
    try:
        smoothing = SmoothingFunction().method1
        bleu1 = sentence_bleu([ref.split()], pred.split(), weights=(1, 0, 0, 0), smoothing_function=smoothing)
        bleu2 = sentence_bleu([ref.split()], pred.split(), weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing)
        bleu3 = sentence_bleu([ref.split()], pred.split(), weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothing)
        bleu4 = sentence_bleu([ref.split()], pred.split(), weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)
    except Exception as e:
        print(f"Error computing BLEU scores: {e}")
        bleu1 = bleu2 = bleu3 = bleu4 = 0.0
    # ROUGE-1 and ROUGE-2
    try:
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True)
        scores = scorer.score(ref, pred)
        rouge1 = float(scores["rouge1"].fmeasure)
        rouge2 = float(scores["rouge2"].fmeasure)
    except Exception as e:
        print(f"Error computing ROUGE scores: {e}")
        rouge1 = rouge2 = 0.0
    # METEOR (robust handling)
    try:
        pred_norm = "" if pred is None else pred.strip().lower()
        ref_norm = "" if ref is None else ref.strip().lower()
        if pred_norm == ref_norm:
            meteor = 1.0
        else:
            meteor = meteor_score([ref_norm.split()], pred_norm.split())
            if meteor is None:
                meteor = 0.0
    except Exception as e:
        print(f"Error computing METEOR score: {e}")
        meteor = 0.0
    return {
        "bertscore": bertscore_value,
        "bleu1": float(bleu1),
        "bleu2": float(bleu2),
        "bleu3": float(bleu3),
        "bleu4": float(bleu4),
        "rouge1": rouge1,
        "rouge2": rouge2,
        "meteor": float(meteor),
    }

# --- APPEND TO JSONL ---
def append_to_jsonl(file_path, new_data):
    try:
        with open(file_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(new_data, ensure_ascii=False) + "\n")
        print(f"Successfully added new document to {file_path}")
    except Exception as e:
        print(f"Error appending to file {file_path}: {e}")

# --- DOWNLOAD FILES AFTER SAVE ---
def download_files_after_save(file_path):
    try:
        target_path = Path('.') / file_path.name
        if target_path.exists():
            target_path = target_path.with_name(target_path.stem + "_new" + target_path.suffix)
        shutil.copy(file_path, target_path)
        print(f"File {file_path} copied for download.")
    except Exception as e:
        print(f"Error copying file {file_path}: {e}")

# --- EXTRACT FINAL ANSWER FROM MODEL THINKING ---
def extract_final_answer(text):
    """
    Extract the final answer from model's thinking process.
    The model often thinks in English then gives final answer in Vietnamese.
    """
    if not text:
        return text
    lines = text.strip().split('\n')
    
    # Look for Vietnamese text patterns that are likely the final answer
    vietnamese_indicators = [
        'là', 'có', 'bằng', 'khoảng', 'từ', 'đến', 'trong', 'vào', 'cao', 'thấp','tăng', 'giảm'
        'tháng', 'năm', 'chiếc', '%', 'triệu', 'tỷ', '°C', 'nghìn', 'mm', 'USD', 'tấn', 
    ]
    
    # Try to find the last line that contains Vietnamese indicators
    for i in range(len(lines)-1, -1, -1):
        line = lines[i].strip()
        if any(indicator in line.lower() for indicator in vietnamese_indicators):
            return line
    for i in range(len(lines)-1, -1, -1):
        if lines[i].strip():
            return lines[i].strip()
    
    return text.strip()

# --- PROCESS SINGLE IMAGE ---
def process_single_document(document, model, processor, results, processed_images, processed_documents):
    document_id = document['document_id']
    if document_id in processed_documents:
        return
    
    image_name = document['image_info']['image_name']
    image_path = images_dir / image_name
    question_answers = document['qa']
    
    for qa in question_answers:
        question = qa['question']
        ground_truth_clean = strip_trailing_punct(qa.get('answer', ""))
        #instruction = qa.get('instruction', "")
        question_prompt = (
            #"\nInstruction: " + (instruction or "") + 
            "Hãy trả lời trực tiếp bằng tiếng Việt. "
            "Chỉ đưa ra câu trả lời cuối cùng, không giải thích. "
            "Câu trả lời phải ngắn gọn, chỉ gồm số và đơn vị (nếu có). "
            "Ví dụ: '35 chiếc', 'Tháng 2', '4,33%'."
            "\nCâu hỏi: " + question
        )
      
        try:
            images, num_patches, original_image = load_image(image_path)
            if images is None:
                print(f"Skipping question for image {image_name} due to image loading failure.")
                continue

            messages = [
                {
                    "role": "user",
                    "content": [{"type": "image", "image": img} for img in images] + [{"type": "text", "text": question_prompt}]
                }
            ]
            # Using processor's chat template
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(
                text=[text],
                images=images,
                padding=True,
                return_tensors="pt"
            ).to("cuda")
            
            # --- GENERATION (keep your existing decoding scheme; adjust here if you want) ---
            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=150,
                    do_sample=True,
                    num_beams=7,
                    temperature=0.4,
                    top_p=0.9,
                    repetition_penalty=1.5
                )

                # trim the input prefix tokens from the outputs for each batch element
                generated_ids_trimmed = [
                    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
                ]
                output_texts = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)
                output_text = output_texts[0].strip()
                # Normalize prediction (remove trailing punctuation) BEFORE metrics
                output_text_clean = strip_trailing_punct(output_text)
            
            # Compute and save metrics (không có Exact Match)
            metrics = compute_metrics(output_text_clean, ground_truth_clean)

            result = {
                "document_id": document_id,
                "image_name": image_name,
                "question": question,
                "ground_truth": ground_truth_clean,
                "prediction": output_text_clean,
                **metrics
            }
            results.append(result)
            append_to_jsonl("results_batch.jsonl", result) 
            download_files_after_save(Path("results_batch.jsonl"))  
            # In output không có Exact Match
            print(f"Image: {image_name} | Predicted Answer: {output_text_clean} | Ground Truth: {ground_truth_clean} | "
                  f"BERTScore: {metrics['bertscore']:.3f} | "
                  f"BLEU-1: {metrics['bleu1']:.3f} | BLEU-2: {metrics['bleu2']:.3f} | BLEU-3: {metrics['bleu3']:.3f} | BLEU-4: {metrics['bleu4']:.3f} | "
                  f"ROUGE-1: {metrics['rouge1']:.3f} | ROUGE-2: {metrics['rouge2']:.3f} | METEOR: {metrics['meteor']:.3f}")
        except Exception as e:
            print(f"Error processing image {image_name} or question: {e}")
            continue

    processed_documents.add(document_id)
    torch.cuda.empty_cache()
    processed_images.append(image_name)


# --- PROCESS ALL DOCUMENTS ---
def process_all_documents_combined(model, processor, jsonl_paths, save_every_image=1, save_path="results_batch.jsonl"):
    documents = []
    for path in jsonl_paths:
        documents += reconstruct_and_load_jsonl(path)
    print(f"Total documents to process: {len(documents)}")
  
    results = []
    processed_images = []
    processed_documents = set()
  
    last_doc_id = None
    if checkpoint_dir.exists():
        checkpoint_data = reconstruct_and_load_jsonl(checkpoint_dir)
        if checkpoint_data:
            try:
                last_doc_id = max(checkpoint_data, key=lambda x: int(x['document_id']))['document_id']
            except Exception:
                last_doc_id = checkpoint_data[-1].get('document_id')
            processed_documents.update(obj['document_id'] for obj in checkpoint_data)
            processed_images.extend(obj.get('image_name', "") for obj in checkpoint_data)
            processed_images = list(sorted(set(processed_images), key=processed_images.index))
            print(f"Resuming from last document_id: {last_doc_id} ({len(processed_images)} images processed)")
        else:
            print(f"Checkpoint file {checkpoint_dir} exists but contains no valid JSON data")
    else:
        print(f"Checkpoint file {checkpoint_dir} does not exist")
    
    start_processing = False
    if last_doc_id:
        try:
            last_doc_num = int(last_doc_id)
            print(f"Will start processing from document_id greater than {last_doc_id}")
        except ValueError:
            print(f"Invalid document_id format in checkpoint: {last_doc_id}. Starting fresh.")
            last_doc_num = -1
    else:
        last_doc_num = -1
    
    batch_count = 0
    for document in documents:
        doc_id = document['document_id']
        try:
            doc_num = int(doc_id)
        except ValueError:
            print(f"Skipping invalid document_id: {doc_id}")
            continue
        if doc_num <= last_doc_num:
            continue
        start_processing = True
        if not start_processing:
            continue
        process_single_document(document, model, processor, results, processed_images, processed_documents)
        batch_count += 1
    
    print("Processing complete, results saved to:", save_path)

# --- AGGREGATE RESULTS ---
def aggregate_results(save_path="results_batch.jsonl"):
    results = []
    try:
        results = reconstruct_and_load_jsonl(save_path)
    except Exception as e:
        print(f"Error reading results file {save_path}: {e}")
        return
  
    n = len(results)
    if n == 0:
        print("No results found in the checkpoint file.")
        return
  
    bertscore_avg = sum(r.get("bertscore", 0.0) for r in results) / n
    bleu1_avg = sum(r.get("bleu1", 0.0) for r in results) / n
    bleu2_avg = sum(r.get("bleu2", 0.0) for r in results) / n
    bleu3_avg = sum(r.get("bleu3", 0.0) for r in results) / n
    bleu4_avg = sum(r.get("bleu4", 0.0) for r in results) / n
    rouge1_avg = sum(r.get("rouge1", 0.0) for r in results) / n
    rouge2_avg = sum(r.get("rouge2", 0.0) for r in results) / n
    meteor_avg = sum(r.get("meteor", 0.0) for r in results) / n
    print(f"Total number of answers: {n}")
    print(f"Average BERTScore: {bertscore_avg:.3f}")
    print(f"Average BLEU-1: {bleu1_avg:.3f}")
    print(f"Average BLEU-2: {bleu2_avg:.3f}")
    print(f"Average BLEU-3: {bleu3_avg:.3f}")
    print(f"Average BLEU-4: {bleu4_avg:.3f}")
    print(f"Average ROUGE-1: {rouge1_avg:.3f}")
    print(f"Average ROUGE-2: {rouge2_avg:.3f}")
    print(f"Average METEOR: {meteor_avg:.3f}")

# --- RUN ---
process_all_documents_combined(model, processor, [chart_jsonl], save_every_image=1, save_path="results_batch.jsonl")
aggregate_results("results_batch.jsonl")

Total documents to process: 395
Checkpoint file results_batch.jsonl does not exist


model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Successfully added new document to results_batch.jsonl
File results_batch.jsonl copied for download.
Image: 0001.png | Predicted Answer: Tháng 1 | Ground Truth: Tháng 2 | BERTScore: 0.970 | BLEU-1: 0.500 | BLEU-2: 0.224 | BLEU-3: 0.174 | BLEU-4: 0.150 | ROUGE-1: 0.667 | ROUGE-2: 0.500 | METEOR: 0.250
Successfully added new document to results_batch.jsonl
File results_batch.jsonl copied for download.
Image: 0001.png | Predicted Answer: Tháng 1 | Ground Truth: Tháng 1 | BERTScore: 1.000 | BLEU-1: 1.000 | BLEU-2: 1.000 | BLEU-3: 0.468 | BLEU-4: 0.316 | ROUGE-1: 1.000 | ROUGE-2: 1.000 | METEOR: 1.000
Successfully added new document to results_batch.jsonl
File results_batch.jsonl copied for download.
Image: 0001.png | Predicted Answer: 47 chiếc | Ground Truth: 35 chiếc | BERTScore: 0.963 | BLEU-1: 0.500 | BLEU-2: 0.224 | BLEU-3: 0.174 | BLEU-4: 0.150 | ROUGE-1: 0.667 | ROUGE-2: 0.500 | METEOR: 0.250
Successfully added new document to results_batch.jsonl
File results_batch.jsonl copied for d

KeyboardInterrupt: 

# Intruction

In [ ]:
import json
from pathlib import Path
from PIL import Image
import torch
from transformers import AutoProcessor, AutoTokenizer
from bert_score import score as bertscore
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
import shutil
import nltk
import re

nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# --- DATA PATHS ---
data_dir = Path('.')
images_dir = data_dir / 'images' / 'images'
chart_jsonl = data_dir / 'Chart_data.jsonl'
#table_jsonl = data_dir / 'Table_data.jsonl'
checkpoint_dir = Path('results_batch.jsonl')

# --- INITIALIZE TOKENIZER / PROCESSOR / MODEL 
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

# --- Normalization utilities
def strip_trailing_punct(s):
    """
    Mimic the Kaggle cell behavior: trim whitespace and remove common trailing punctuation.
    """
    if s is None:
        return ""
    s = str(s).strip()
    # Remove surrounding brackets if present
    s = s.strip('()[]{}')
    # Remove trailing punctuation including some fullwidth punctuation and invisible marks
    s = s.rstrip(" .，。,;:!?\"'\u3000\uFEFF")
    return s

def normalize_whitespace(s):
    """
    Collapse repeated whitespace (including NBSP and zero-width spaces) into a single space,
    and trim leading/trailing whitespace.
    """
    if s is None:
        return ""
    s = str(s)
    s = s.replace('\u00A0', ' ')  # NBSP
    s = s.replace('\u200b', '')   # zero width space
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def clean_text_for_metrics(s):
    """
    Combine whitespace normalization and trailing-punctuation stripping,
    returning a normalized string suitable for metrics computation.
    """
    if s is None:
        return ""
    s = normalize_whitespace(s)
    s = strip_trailing_punct(s)
    s = normalize_whitespace(s)
    return s

# --- READ JSONL FILE ---
def reconstruct_and_load_jsonl(file_path):
    data = []
    current_json = ""
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            current_json += line
            try:
                json_obj = json.loads(current_json)
                data.append(json_obj)
                current_json = ""
            except json.JSONDecodeError:
                continue
    return data

# --- IMAGE PREPROCESSING ---
def find_closest_aspect_ratio(aspect_ratio, target_ratios):
    min_diff = float('inf')
    best_ratio = (1, 1)
    for ratio in target_ratios:
        target_ar = ratio[0] / ratio[1]
        diff = abs(aspect_ratio - target_ar)
        if diff < min_diff:
            min_diff = diff
            best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, min_num=1, max_num=6, image_size=448, use_thumbnail=False):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / orig_height
    target_ratios = set()
    for n in range(min_num, max_num + 1):
        for i in range(1, n + 1):
            for j in range(1, n + 1):
                if i * j <= max_num and i * j >= min_num:
                    target_ratios.add((i, j))
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_aspect_ratio = find_closest_aspect_ratio(aspect_ratio, target_ratios)
    w_ratio, h_ratio = target_aspect_ratio
    target_width = int(image_size * w_ratio)
    target_height = int(image_size * h_ratio)
    num_patches = w_ratio * h_ratio
    resized_img = image.resize((target_width, target_height), Image.Resampling.BICUBIC)
    processed_images = []
    for i in range(int(h_ratio)):
        for j in range(int(w_ratio)):
            left = j * image_size
            upper = i * image_size
            right = left + image_size
            lower = upper + image_size
            patch = resized_img.crop((left, upper, right, lower))
            processed_images.append(patch)
    if use_thumbnail and num_patches != 1:
        thumbnail_img = image.resize((image_size, image_size), Image.Resampling.BICUBIC)
        processed_images.append(thumbnail_img)
    return processed_images, num_patches

def load_image(image_file, input_size=448, max_num=6):
    try:
        image = Image.open(image_file).convert('RGB')
        width, height = 448, 448
        image = image.resize((width, height))
        images, num_patches = dynamic_preprocess(image, image_size=input_size, use_thumbnail=True, max_num=max_num)
        return images, num_patches, image
    except Exception as e:
        print(f"Error loading image {image_file}: {e}")
        return None, None, None

# --- CALCULATE METRICS ---
def compute_metrics(pred, ref):
    # Clean both prediction and reference of trailing punctuation and extra whitespace
    pred = "" if pred is None else strip_trailing_punct(pred)
    ref = "" if ref is None else strip_trailing_punct(ref)
    # BERTScore
    try:
        P, R, F1 = bertscore([pred], [ref], lang="vi", rescale_with_baseline=False)
        bertscore_value = float(F1[0])
    except Exception as e:
        print(f"Error computing BERTScore: {e}")
        bertscore_value = 0.0
    # BLEU-1 to BLEU-4
    try:
        smoothing = SmoothingFunction().method1
        bleu1 = sentence_bleu([ref.split()], pred.split(), weights=(1, 0, 0, 0), smoothing_function=smoothing)
        bleu2 = sentence_bleu([ref.split()], pred.split(), weights=(0.5, 0.5, 0, 0), smoothing_function=smoothing)
        bleu3 = sentence_bleu([ref.split()], pred.split(), weights=(0.33, 0.33, 0.33, 0), smoothing_function=smoothing)
        bleu4 = sentence_bleu([ref.split()], pred.split(), weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smoothing)
    except Exception as e:
        print(f"Error computing BLEU scores: {e}")
        bleu1 = bleu2 = bleu3 = bleu4 = 0.0
    # ROUGE-1 and ROUGE-2
    try:
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2'], use_stemmer=True)
        scores = scorer.score(ref, pred)
        rouge1 = float(scores["rouge1"].fmeasure)
        rouge2 = float(scores["rouge2"].fmeasure)
    except Exception as e:
        print(f"Error computing ROUGE scores: {e}")
        rouge1 = rouge2 = 0.0
    # METEOR (robust handling)
    try:
        pred_norm = "" if pred is None else pred.strip().lower()
        ref_norm = "" if ref is None else ref.strip().lower()
        if pred_norm == ref_norm:
            meteor = 1.0
        else:
            meteor = meteor_score([ref_norm.split()], pred_norm.split())
            if meteor is None:
                meteor = 0.0
    except Exception as e:
        print(f"Error computing METEOR score: {e}")
        meteor = 0.0
    return {
        "bertscore": bertscore_value,
        "bleu1": float(bleu1),
        "bleu2": float(bleu2),
        "bleu3": float(bleu3),
        "bleu4": float(bleu4),
        "rouge1": rouge1,
        "rouge2": rouge2,
        "meteor": float(meteor),
    }

# --- APPEND TO JSONL ---
def append_to_jsonl(file_path, new_data):
    try:
        with open(file_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(new_data, ensure_ascii=False) + "\n")
        print(f"Successfully added new document to {file_path}")
    except Exception as e:
        print(f"Error appending to file {file_path}: {e}")

# --- DOWNLOAD FILES AFTER SAVE ---
def download_files_after_save(file_path):
    try:
        target_path = Path('.') / file_path.name
        if target_path.exists():
            target_path = target_path.with_name(target_path.stem + "_new" + target_path.suffix)
        shutil.copy(file_path, target_path)
        print(f"File {file_path} copied for download.")
    except Exception as e:
        print(f"Error copying file {file_path}: {e}")

# --- EXTRACT FINAL ANSWER FROM MODEL THINKING ---
def extract_final_answer(text):
    """
    Extract the final answer from model's thinking process.
    The model often thinks in English then gives final answer in Vietnamese.
    """
    if not text:
        return text
    lines = text.strip().split('\n')
    
    # Look for Vietnamese text patterns that are likely the final answer
    vietnamese_indicators = [
        'là', 'có', 'bằng', 'khoảng', 'từ', 'đến', 'trong', 'vào', 'cao', 'thấp','tăng', 'giảm'
        'tháng', 'năm', 'chiếc', '%', 'triệu', 'tỷ', '°C', 'nghìn', 'mm', 'USD', 'tấn', 
    ]
    
    # Try to find the last line that contains Vietnamese indicators
    for i in range(len(lines)-1, -1, -1):
        line = lines[i].strip()
        if any(indicator in line.lower() for indicator in vietnamese_indicators):
            return line
    for i in range(len(lines)-1, -1, -1):
        if lines[i].strip():
            return lines[i].strip()
    
    return text.strip()

# --- PROCESS SINGLE IMAGE ---
def process_single_document(document, model, processor, results, processed_images, processed_documents):
    document_id = document['document_id']
    if document_id in processed_documents:
        return
    
    image_name = document['image_info']['image_name']
    image_path = images_dir / image_name
    question_answers = document['qa']
    
    for qa in question_answers:
        question = qa['question']
        ground_truth_clean = strip_trailing_punct(qa.get('answer', ""))
        instruction = qa.get('instruction', "")
        question_prompt = (
            "\nInstruction: " + (instruction or "") + 
            "Hãy trả lời trực tiếp bằng tiếng Việt. "
            "Chỉ đưa ra câu trả lời cuối cùng, không giải thích. "
            "Câu trả lời phải ngắn gọn, chỉ gồm số và đơn vị (nếu có). "
            "Ví dụ: '35 chiếc', 'Tháng 2', '4,33%'."
            "\nCâu hỏi: " + question
        )
      
        try:
            images, num_patches, original_image = load_image(image_path)
            if images is None:
                print(f"Skipping question for image {image_name} due to image loading failure.")
                continue

            messages = [
                {
                    "role": "user",
                    "content": [{"type": "image", "image": img} for img in images] + [{"type": "text", "text": question_prompt}]
                }
            ]
            # Using processor's chat template
            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = processor(
                text=[text],
                images=images,
                padding=True,
                return_tensors="pt"
            ).to("cuda")
            
            # --- GENERATION (keep your existing decoding scheme; adjust here if you want) ---
            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=150,
                    do_sample=True,
                    num_beams=7,
                    temperature=0.4,
                    top_p=0.9,
                    repetition_penalty=1.5
                )

                # trim the input prefix tokens from the outputs for each batch element
                generated_ids_trimmed = [
                    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
                ]
                output_texts = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True)
                output_text = output_texts[0].strip()
                # Normalize prediction (remove trailing punctuation) BEFORE metrics
                output_text_clean = strip_trailing_punct(output_text)
            
            # Compute and save metrics (không có Exact Match)
            metrics = compute_metrics(output_text_clean, ground_truth_clean)

            result = {
                "document_id": document_id,
                "image_name": image_name,
                "question": question,
                "ground_truth": ground_truth_clean,
                "prediction": output_text_clean,
                **metrics
            }
            results.append(result)
            append_to_jsonl("results_batch.jsonl", result) 
            download_files_after_save(Path("results_batch.jsonl"))  
            # In output không có Exact Match
            print(f"Image: {image_name} | Predicted Answer: {output_text_clean} | Ground Truth: {ground_truth_clean} | "
                  f"BERTScore: {metrics['bertscore']:.3f} | "
                  f"BLEU-1: {metrics['bleu1']:.3f} | BLEU-2: {metrics['bleu2']:.3f} | BLEU-3: {metrics['bleu3']:.3f} | BLEU-4: {metrics['bleu4']:.3f} | "
                  f"ROUGE-1: {metrics['rouge1']:.3f} | ROUGE-2: {metrics['rouge2']:.3f} | METEOR: {metrics['meteor']:.3f}")
        except Exception as e:
            print(f"Error processing image {image_name} or question: {e}")
            continue

    processed_documents.add(document_id)
    torch.cuda.empty_cache()
    processed_images.append(image_name)


# --- PROCESS ALL DOCUMENTS ---
def process_all_documents_combined(model, processor, jsonl_paths, save_every_image=1, save_path="results_batch.jsonl"):
    documents = []
    for path in jsonl_paths:
        documents += reconstruct_and_load_jsonl(path)
    print(f"Total documents to process: {len(documents)}")
  
    results = []
    processed_images = []
    processed_documents = set()
  
    last_doc_id = None
    if checkpoint_dir.exists():
        checkpoint_data = reconstruct_and_load_jsonl(checkpoint_dir)
        if checkpoint_data:
            try:
                last_doc_id = max(checkpoint_data, key=lambda x: int(x['document_id']))['document_id']
            except Exception:
                last_doc_id = checkpoint_data[-1].get('document_id')
            processed_documents.update(obj['document_id'] for obj in checkpoint_data)
            processed_images.extend(obj.get('image_name', "") for obj in checkpoint_data)
            processed_images = list(sorted(set(processed_images), key=processed_images.index))
            print(f"Resuming from last document_id: {last_doc_id} ({len(processed_images)} images processed)")
        else:
            print(f"Checkpoint file {checkpoint_dir} exists but contains no valid JSON data")
    else:
        print(f"Checkpoint file {checkpoint_dir} does not exist")
    
    start_processing = False
    if last_doc_id:
        try:
            last_doc_num = int(last_doc_id)
            print(f"Will start processing from document_id greater than {last_doc_id}")
        except ValueError:
            print(f"Invalid document_id format in checkpoint: {last_doc_id}. Starting fresh.")
            last_doc_num = -1
    else:
        last_doc_num = -1
    
    batch_count = 0
    for document in documents:
        doc_id = document['document_id']
        try:
            doc_num = int(doc_id)
        except ValueError:
            print(f"Skipping invalid document_id: {doc_id}")
            continue
        if doc_num <= last_doc_num:
            continue
        start_processing = True
        if not start_processing:
            continue
        process_single_document(document, model, processor, results, processed_images, processed_documents)
        batch_count += 1
    
    print("Processing complete, results saved to:", save_path)

# --- AGGREGATE RESULTS ---
def aggregate_results(save_path="results_batch.jsonl"):
    results = []
    try:
        results = reconstruct_and_load_jsonl(save_path)
    except Exception as e:
        print(f"Error reading results file {save_path}: {e}")
        return
  
    n = len(results)
    if n == 0:
        print("No results found in the checkpoint file.")
        return
  
    bertscore_avg = sum(r.get("bertscore", 0.0) for r in results) / n
    bleu1_avg = sum(r.get("bleu1", 0.0) for r in results) / n
    bleu2_avg = sum(r.get("bleu2", 0.0) for r in results) / n
    bleu3_avg = sum(r.get("bleu3", 0.0) for r in results) / n
    bleu4_avg = sum(r.get("bleu4", 0.0) for r in results) / n
    rouge1_avg = sum(r.get("rouge1", 0.0) for r in results) / n
    rouge2_avg = sum(r.get("rouge2", 0.0) for r in results) / n
    meteor_avg = sum(r.get("meteor", 0.0) for r in results) / n
    print(f"Total number of answers: {n}")
    print(f"Average BERTScore: {bertscore_avg:.3f}")
    print(f"Average BLEU-1: {bleu1_avg:.3f}")
    print(f"Average BLEU-2: {bleu2_avg:.3f}")
    print(f"Average BLEU-3: {bleu3_avg:.3f}")
    print(f"Average BLEU-4: {bleu4_avg:.3f}")
    print(f"Average ROUGE-1: {rouge1_avg:.3f}")
    print(f"Average ROUGE-2: {rouge2_avg:.3f}")
    print(f"Average METEOR: {meteor_avg:.3f}")

# --- RUN ---
process_all_documents_combined(model, processor, [chart_jsonl], save_every_image=1, save_path="results_batch.jsonl")
aggregate_results("results_batch.jsonl")

Total documents to process: 395
Resuming from last document_id: 0003 (3 images processed)
Will start processing from document_id greater than 0003
Successfully added new document to results_batch.jsonl
File results_batch.jsonl copied for download.
Image: 0004.png | Predicted Answer: 30 tỉ đồng | Ground Truth: 30 tỉ đồng | BERTScore: 1.000 | BLEU-1: 1.000 | BLEU-2: 1.000 | BLEU-3: 1.000 | BLEU-4: 0.562 | ROUGE-1: 1.000 | ROUGE-2: 1.000 | METEOR: 1.000
Successfully added new document to results_batch.jsonl
File results_batch.jsonl copied for download.
Image: 0004.png | Predicted Answer: 34 tỉ đồng | Ground Truth: 34 tỉ đồng | BERTScore: 1.000 | BLEU-1: 1.000 | BLEU-2: 1.000 | BLEU-3: 1.000 | BLEU-4: 0.562 | ROUGE-1: 1.000 | ROUGE-2: 1.000 | METEOR: 1.000
Successfully added new document to results_batch.jsonl
File results_batch.jsonl copied for download.
Image: 0004.png | Predicted Answer: 38 tỉ đồng | Ground Truth: 38 tỉ đồng | BERTScore: 1.000 | BLEU-1: 1.000 | BLEU-2: 1.000 | BLEU-3: 1